# YOLOv5 Training Pipeline — MOT17 Anomaly Detection Project

**⚠️ Record-only notebook.** This notebook documents the full training pipeline that produced the trained weights (already saved to Google Drive). It is kept for the git repo / report as a reference and is **not meant to be re-run** end-to-end. For loading trained models and running inference/tracking, use `inference_and_tracking.ipynb` instead.

Pipeline covered here:
1. Kaggle setup + dataset download (MOT17, CUHK Avenue)
2. Convert MOT17 ground truth to YOLO format
3. Train 3 YOLOv5 variants with `train.py` (yolov5s, yolov5m, yolov5m-frozen) and compare
4. Train a 4th variant (`yolov5mu`) via the `ultralytics` Python API
5. Save best weights to Google Drive

## 1. Kaggle setup & dataset download

In [ ]:
from google.colab import files
files.upload('kaggle.json')

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
mot17_path = kagglehub.dataset_download("wenhoujinjust/mot-17")
print(mot17_path)

In [ ]:
avenue_path = kagglehub.dataset_download("hihnguynth/cuhk-avenue-dataset")
print(avenue_path)

In [ ]:
import subprocess
print(subprocess.run(["find", mot17_path, "-maxdepth", "3"], capture_output=True, text=True).stdout)

## 2. Convert MOT17 ground truth (MOT format) → YOLO format

In [ ]:
import configparser
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

MOT17_ROOT = Path(mot17_path) / "MOT17" / "train"
OUTPUT_ROOT = Path("/kaggle/working/mot17_yolo")
DETECTOR_VARIANT = "FRCNN"   # keep one variant per sequence, ignore DPM/SDP
VAL_SPLIT = 0.15
SEED = 42
random.seed(SEED)

assert MOT17_ROOT.exists(), f"Can't find {MOT17_ROOT} -- check the dataset mounted correctly"

In [ ]:
def read_seqinfo(seq_dir: Path):
    """Read imWidth / imHeight from seqinfo.ini (needed to normalize boxes)."""
    cfg = configparser.ConfigParser()
    cfg.read(seq_dir / "seqinfo.ini")
    width = int(cfg["Sequence"]["imWidth"])
    height = int(cfg["Sequence"]["imHeight"])
    return width, height


def convert_to_yolo(size, box):
    """
    size = (image_width, image_height)
    box  = (top_left_x, top_left_y, width, height)   <- MOT17 format
    returns (x_center, y_center, width, height), normalized 0-1
    """
    dw = 1.0 / size[0]
    dh = 1.0 / size[1]
    x_center = (box[0] + box[2] / 2.0) * dw
    y_center = (box[1] + box[3] / 2.0) * dh
    w = box[2] * dw
    h = box[3] * dh
    return x_center, y_center, w, h


def parse_gt(seq_dir: Path):
    """Parse gt/gt.txt -> {frame_id: [(x, y, w, h), ...]}.
    Keeps only class=1 (pedestrian) with visibility > 0.2."""
    gt_path = seq_dir / "gt" / "gt.txt"
    frames = {}
    with open(gt_path, "r") as f:
        for line in f:
            parts = line.strip().split(",")
            frame_id, _id = int(parts[0]), int(parts[1])
            x, y, w, h = map(float, parts[2:6])
            conf, cls, vis = float(parts[6]), int(parts[7]), float(parts[8])
            if cls != 1 or vis <= 0.2 or conf == 0:
                continue
            frames.setdefault(frame_id, []).append((x, y, w, h))
    return frames


def process_sequence(seq_dir: Path, image_out: Path, label_out: Path):
    """Convert one MOT17-XX-FRCNN sequence into YOLO images+labels."""
    width, height = read_seqinfo(seq_dir)
    frames = parse_gt(seq_dir)
    img_dir = seq_dir / "img1"

    written = 0
    for frame_id, boxes in frames.items():
        img_name = f"{frame_id:06d}.jpg"
        src_img = img_dir / img_name
        if not src_img.exists():
            continue
        new_stem = f"{seq_dir.name}_{frame_id:06d}"
        shutil.copy(src_img, image_out / f"{new_stem}.jpg")
        with open(label_out / f"{new_stem}.txt", "w") as lf:
            for box in boxes:
                xc, yc, w, h = convert_to_yolo((width, height), box)
                lf.write(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
        written += 1
    return written


In [ ]:
sequences = sorted(
    d for d in MOT17_ROOT.iterdir()
    if d.is_dir() and d.name.endswith(DETECTOR_VARIANT)
)
print(f"Found {len(sequences)} sequences using detector variant '{DETECTOR_VARIANT}':")
for s in sequences:
    print("  -", s.name)

for split in ("train", "val"):
    (OUTPUT_ROOT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / split / "labels").mkdir(parents=True, exist_ok=True)

tmp_img = OUTPUT_ROOT / "_all" / "images"
tmp_lbl = OUTPUT_ROOT / "_all" / "labels"
tmp_img.mkdir(parents=True, exist_ok=True)
tmp_lbl.mkdir(parents=True, exist_ok=True)

total = 0
for seq in sequences:
    n = process_sequence(seq, tmp_img, tmp_lbl)
    print(f"  {seq.name}: {n} frames processed")
    total += n
print(f"Total frames converted: {total}")


In [ ]:
all_images = sorted(tmp_img.glob("*.jpg"))
random.shuffle(all_images)
n_val = int(len(all_images) * VAL_SPLIT)
val_set = set(all_images[:n_val])

for img_path in all_images:
    split = "val" if img_path in val_set else "train"
    lbl_path = tmp_lbl / (img_path.stem + ".txt")
    shutil.move(str(img_path), OUTPUT_ROOT / split / "images" / img_path.name)
    shutil.move(str(lbl_path), OUTPUT_ROOT / split / "labels" / lbl_path.name)

shutil.rmtree(OUTPUT_ROOT / "_all")
print(f"Done. Train: {len(all_images) - n_val} frames | Val: {n_val} frames")
print(f"Output written to: {OUTPUT_ROOT.resolve()}")


### Sanity check: visualize a converted sample

In [ ]:
sample_img_path = sorted((OUTPUT_ROOT / "train" / "images").glob("*.jpg"))[0]
sample_lbl_path = OUTPUT_ROOT / "train" / "labels" / (sample_img_path.stem + ".txt")

img = Image.open(sample_img_path)
w, h = img.size

fig, ax = plt.subplots(1, figsize=(10, 6))
ax.imshow(img)

with open(sample_lbl_path) as f:
    for line in f:
        cls, xc, yc, bw, bh = map(float, line.split())
        x0 = (xc - bw / 2) * w
        y0 = (yc - bh / 2) * h
        rect = patches.Rectangle((x0, y0), bw * w, bh * h,
                                  linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)

ax.set_title(sample_img_path.name)
plt.axis("off")
plt.show()


## 3. Train YOLOv5 variants (via `train.py`, official ultralytics/yolov5 repo)

In [ ]:
!git clone https://github.com/ultralytics/yolov5 /kaggle/working/yolov5
%cd /kaggle/working/yolov5
!pip install -qr requirements.txt


In [ ]:
data_yaml = """
train: /kaggle/working/mot17_yolo/train/images
val: /kaggle/working/mot17_yolo/val/images

nc: 1
names: ["person"]
"""

with open("/kaggle/working/mot17.yaml", "w") as f:
    f.write(data_yaml)

print(data_yaml)


In [ ]:

!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 5 \
  --data /kaggle/working/mot17.yaml \
  --weights yolov5s.pt \
  --name mot17_yolov5s


In [ ]:
!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 5 \
  --data /kaggle/working/mot17.yaml \
  --weights yolov5m.pt \
  --name mot17_yolov5m


In [ ]:
!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 5 \
  --data /kaggle/working/mot17.yaml \
  --weights yolov5m.pt \
  --freeze 10 \
  --name mot17_yolov5m_frozen


### Compare the 3 variants

In [ ]:
import pandas as pd

runs = {
    "YOLOv5s": "runs/train/mot17_yolov5s/results.csv",
    "YOLOv5m": "runs/train/mot17_yolov5m/results.csv",
    "YOLOv5m-Frozen": "runs/train/mot17_yolov5m_frozen/results.csv",
}

rows = []
for name, path in runs.items():
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    last = df.iloc[-1]
    rows.append({
        "Model": name,
        "Precision": round(last["metrics/precision"], 3),
        "Recall": round(last["metrics/recall"], 3),
        "mAP@0.5": round(last["metrics/mAP_0.5"], 3),
    })

comparison = pd.DataFrame(rows)
comparison


In [ ]:
# Save the comparison table and a bar chart -- useful for the README and the PPT later
import matplotlib.pyplot as plt

comparison.to_csv("/kaggle/working/detection_comparison.csv", index=False)

ax = comparison.set_index("Model")[["Precision", "Recall", "mAP@0.5"]].plot(
    kind="bar", figsize=(8, 5), rot=0
)
ax.set_title("YOLOv5 Variant Comparison on MOT17 (5 epochs)")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("/kaggle/working/detection_comparison.png", dpi=150)
plt.show()


In [ ]:
BEST_RUN = "mot17_yolov5m_frozen"   # change if a different variant wins on your run

import shutil
shutil.copy(f"runs/train/{BEST_RUN}/weights/best.pt", "/kaggle/working/best.pt")
print("Best weights copied to /kaggle/working/best.pt")


## 4. Train a 4th variant — `yolov5mu` via the `ultralytics` Python API
This is the run that ultimately produced the best result (Frozen-YOLOv5m, mAP@0.5 = 0.773) referenced in the report — kept here as a second, simpler training path using the `ultralytics` package directly instead of `train.py`.

In [ ]:
!pip install -q ultralytics


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/anomaly_detection_SOC_26"

In [ ]:
import kagglehub
mot17_path = kagglehub.dataset_download("wenhoujinjust/mot-17")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov5mu.pt")  # 'u' = updated/anchor-free YOLOv5m, native ultralytics format
model.train(
    data="/kaggle/working/mot17.yaml",
    epochs=5,
    imgsz=640,
    batch=16,
    name="mot17_yolov5m_v2",
)

### Save best weights to Google Drive

In [ ]:
import shutil
shutil.copy("runs/detect/mot17_yolov5m_v2/weights/best.pt", f"{SAVE_DIR}/mot17_yolov5m_v2_best.pt")